# Building Multimodal RAG Application — Part 6: LVLM Inference

This Colab-ready companion notebook recreates the article's five Large Vision-Language Model (LVLM) inference patterns with the current Hugging Face Transformers `image-text-to-text` pipeline. It defaults to the lightweight open `HuggingFaceTB/SmolVLM-500M-Instruct` checkpoint so the examples can be executed and verified without a high-memory GPU.

> **Model note:** The archived article uses LLaVA-1.5-7B through Prediction Guard. Set `MODEL_ID = "llava-hf/llava-1.5-7b-hf"` on a sufficiently large GPU to reproduce that model choice. The lightweight default exercises the same current interface and use cases.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=4.52" accelerate bitsandbytes pillow requests

## 2. Configuration and sample data

The first image mirrors the article's skateboard example. The diagram image from the Hugging Face documentation gives us a reliable textual-cue example. Replace either URL with frames retrieved by your own multimodal RAG pipeline.

In [ ]:
from io import BytesIO

import requests
from PIL import Image
from IPython.display import display

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
RUN_MODEL = True

skateboard_url = (
    "https://farm4.staticflickr.com/3300/3497460990_11dfb95dd1_z.jpg"
)
diagram_url = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/transformers/tasks/ai2d-demo.jpg"
)

astronaut_url = (
    "https://cdn-images-1.medium.com/max/800/"
    "1*Jlado1XvMOgrgGXYVD1wQg.png"
)
astronaut_transcript = (
    "As I look back on the mission that we've had here on the International "
    "Space Station, I'm proud to have been a part of much of the science "
    "activities that happened over the last two months."
)

def load_image(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

skateboard_image = load_image(skateboard_url)
diagram_image = load_image(diagram_url)
astronaut_image = load_image(astronaut_url)
display(skateboard_image)
display(diagram_image)
display(astronaut_image)

## 3. Load LLaVA through the current Transformers interface

The original December 2024 article used a hosted Prediction Guard endpoint. This notebook calls the open checkpoint directly through Transformers while preserving the same tutorial concepts.

In [ ]:
import torch
from transformers import pipeline

generator = None
if RUN_MODEL:
    generator = pipeline(
        task="image-text-to-text",
        model=MODEL_ID,
        device_map="auto",
        dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
    )
else:
    print("Model loading skipped. Set RUN_MODEL = True to run inference.")

In [ ]:
def ask_lvlm(messages, max_new_tokens=120):
    if generator is None:
        print("Preview messages:")
        for message in messages:
            print(message)
        return "[Set RUN_MODEL = True to generate a response.]"

    result = generator(
        text=messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    generated = result[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1]["content"]
    return generated

## 4. Image captioning

In [ ]:
caption_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": skateboard_url},
            {
                "type": "text",
                "text": (
                    "In two sentences, describe only clearly visible details. "
                    "Focus on the person's pose, skateboard, and concrete skate park; "
                    "do not invent scenery outside the frame."
                ),
            },
        ],
    }
]
caption = ask_lvlm(caption_messages)
print(caption)

## 5. Visual question answering

In [ ]:
visual_qa_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": skateboard_url},
            {"type": "text", "text": "What is likely going to happen next?"},
        ],
    }
]
visual_answer = ask_lvlm(visual_qa_messages)
print(visual_answer)

## 6. Question answering over textual cues in an image

In [ ]:
text_cue_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": diagram_url},
            {"type": "text", "text": "What does the label 15 represent?"},
        ],
    }
]
text_cue_answer = ask_lvlm(text_cue_messages)
print(text_cue_answer)

## 7. Transcript-grounded question answering

Retrieved captions and transcripts can be inserted alongside the image. This supplies evidence that may not be visible in the pixels alone.

In [ ]:
transcript_prompt = (
    f"The transcript associated with the image is: '{astronaut_transcript}'\n"
    "What does the speaker feel about the crew's work? "
    "Answer using the supplied transcript and explain your evidence."
)
transcript_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": astronaut_url},
            {"type": "text", "text": transcript_prompt},
        ],
    }
]
transcript_answer = ask_lvlm(transcript_messages)
print(transcript_answer)

## 8. Multi-turn question answering

Append the first model response and a follow-up question to preserve conversational context.

In [ ]:
multi_turn_messages = transcript_messages + [
    {
        "role": "assistant",
        "content": [{"type": "text", "text": transcript_answer}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Where did the crew complete this work?"}
        ],
    },
]
follow_up_answer = ask_lvlm(multi_turn_messages)
print(follow_up_answer)

## 9. Connect this stage to multimodal RAG

In a complete application, replace the sample URL and transcript with the image path and neighboring transcript returned by the vector-store retrieval stage from Part 5. The LVLM then receives the user's question plus the retrieved multimodal evidence, and produces the final grounded response.